In [1]:
import pandas as pd
import html
import re
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

In [2]:
df_train = pd.read_csv("../dataset/drugsComTrain_raw.csv", delimiter=',')
df_test  = pd.read_csv("../dataset/drugsComTest_raw.csv", delimiter=',')

In [3]:
df_train.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37


In [4]:
df_test.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,163740,Mirtazapine,Depression,"""I&#039;ve tried a few antidepressants over th...",10,28-Feb-12,22
1,206473,Mesalamine,"Crohn's Disease, Maintenance","""My son has Crohn&#039;s disease and has done ...",8,17-May-09,17
2,159672,Bactrim,Urinary Tract Infection,"""Quick reduction of symptoms""",9,29-Sep-17,3
3,39293,Contrave,Weight Loss,"""Contrave combines drugs that were used for al...",9,5-Mar-17,35
4,97768,Cyclafem 1 / 35,Birth Control,"""I have been on this birth control for one cyc...",9,22-Oct-15,4


In [5]:
print("Train dataset shape:", df_train.shape)
print("Test dataset shape:", df_test.shape)
print("Train proportion:", df_train.shape[0] / (df_test.shape[0] + df_train.shape[0]))

Train dataset shape: (161297, 7)
Test dataset shape: (53766, 7)
Train proportion: 0.7499988375499272


In [6]:
df_train_reg = df_train[["review", "rating"]]
df_test_reg = df_test[["review", "rating"]]

In [7]:
def clean_text(text):
    # Decodify HTML (&#039; → ')
    text = html.unescape(text)
    
    # Lowercase
    text = text.lower()
    
    # Mantain letters and important punctuation (apostrophes)
    text = re.sub(r"[^a-z\s']", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [8]:
df_train_reg.head()

,review,rating
0,"""It has no side effect, I take it in combinati...",9
1,"""My son is halfway through his fourth week of ...",8
2,"""I used to take another oral contraceptive, wh...",5
3,"""This is my first time using any form of birth...",8
4,"""Suboxone has completely turned my life around...",9


In [9]:
df_train_reg["review"] = df_train_reg["review"].apply(clean_text)
df_test_reg["review"] = df_test_reg["review"].apply(clean_text)

df_test_reg.head()

C:\Users\PC\AppData\Local\Temp\ipykernel_6328\349713204.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_reg["review"] = df_train_reg["review"].apply(clean_text)
C:\Users\PC\AppData\Local\Temp\ipykernel_6328\349713204.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_reg["review"] = df_test_reg["review"].apply(clean_text)


,review,rating
0,i've tried a few antidepressants over the year...,10
1,my son has crohn's disease and has done very w...,8
2,quick reduction of symptoms,9
3,contrave combines drugs that were used for alc...,9
4,i have been on this birth control for one cycl...,9


In [10]:
stop_words = set(text.ENGLISH_STOP_WORDS)
negations = {"no", "not", "never"}
custom_stop_words = stop_words - negations

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    stop_words=list(custom_stop_words),
    max_features=20000
)

In [14]:
from tqdm import tqdm

print("Vectorizing X_train...")
X_train = vectorizer.fit_transform(tqdm(df_train_reg["review"]))

print("\nVectorizing X_test...")
X_test  = vectorizer.transform(tqdm(df_test_reg["review"]))

y_train = df_train_reg["rating"]
y_test  = df_test_reg["rating"]

# Print shapes of the resulting matrices
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Vectorizing X_train...


100%|██████████| 161297/161297 [00:13<00:00, 11612.12it/s]



Vectorizing X_test...


100%|██████████| 53766/53766 [00:04<00:00, 12131.95it/s]



X_train shape: (161297, 20000)
X_test shape: (53766, 20000)


In [18]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import numpy as np

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# Maintain predictions in range 1–10
y_pred_scale = np.clip(y_pred, 1, 10)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_scale))

print(f"RMSE: {rmse:.4f}")

RMSE: 2.1610
